In [ ]:
import pandas as pd
import numpy as np
from datetime import date, datetime
import os

In [ ]:
# Determine data path
if os.path.exists('/workspace/data/'):
    DATA_DIR = '/workspace/data/'
else:
    DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'environment', 'data') + '/'
    if not os.path.exists(DATA_DIR):
        DATA_DIR = '../environment/data/'

# Read the assumptions CSV
csv_path = os.path.join(DATA_DIR, 'assumptions_export.csv')
if os.path.exists(csv_path):
    raw = pd.read_csv(csv_path, header=None)
else:
    # Try Excel
    xlsx_path = os.path.join(DATA_DIR, 'MO17-Round-1-Sec-3-When-it-Rains-it-Pours-data.xlsx')
    raw = pd.read_excel(xlsx_path, header=None)

print('Data loaded, shape:', raw.shape)

In [ ]:
# =====================================================
# DEFINE SEMI-ANNUAL PERIODS
# =====================================================
# Period 0: Dec 2017 (initial balances)
# Periods 1-36: Jun 2018 through Dec 2035

period_ends = []
period_starts = []

# Period 0: Dec 2017
period_starts.append(date(2017, 12, 1))
period_ends.append(date(2017, 12, 31))

# Periods 1-36
for year in range(2018, 2036):
    # H1: Jan 1 to Jun 30
    period_starts.append(date(year, 1, 1))
    period_ends.append(date(year, 6, 30))
    # H2: Jul 1 to Dec 31
    period_starts.append(date(year, 7, 1))
    period_ends.append(date(year, 12, 31))

n_periods = len(period_ends)  # 37 (0 to 36)
print(f'Total periods: {n_periods} (period 0 = Dec 2017, periods 1-36 = Jun 2018 to Dec 2035)')
print(f'First period end: {period_ends[0]}, Last period end: {period_ends[-1]}')

In [ ]:
# =====================================================
# SENIOR 1 LOAN
# =====================================================
# Opening balance: $10,000,000 on Dec 31, 2017
# Equal principal repayments, final repayment Dec 31, 2023
# That's periods 1 through 12 (Jun 2018 to Dec 2023) = 12 periods

s1_opening = 10_000_000.0
s1_n_repayments = 12  # Jun 2018 through Dec 2023
s1_principal_repayment = s1_opening / s1_n_repayments

# Interest rate schedule (dates are inclusive on both From and To)
s1_rate_schedule = [
    (date(2018, 1, 1), date(2019, 12, 14), 0.0435),
    (date(2019, 12, 15), date(2021, 4, 4), 0.055),
    (date(2021, 4, 5), date(2023, 10, 24), 0.058),
    (date(2023, 10, 25), date(2023, 12, 31), 0.062),
]

def compute_s1_interest_rate(p_start, p_end, rate_schedule):
    """
    Compute effective semi-annual interest rate for Senior 1.
    For each rate in the schedule, count the number of days it applies
    within the period [p_start, p_end], multiply by rate/365.
    """
    total_rate = 0.0
    for r_start, r_end, annual_rate in rate_schedule:
        # Overlap of [p_start, p_end] and [r_start, r_end]
        overlap_start = max(p_start, r_start)
        overlap_end = min(p_end, r_end)
        if overlap_start <= overlap_end:
            days = (overlap_end - overlap_start).days + 1
            total_rate += annual_rate * days / 365
    return total_rate

# Build Senior 1 schedule
s1_balance = np.zeros(n_periods)
s1_interest_payment = np.zeros(n_periods)
s1_principal_payment = np.zeros(n_periods)
s1_interest_rate = np.zeros(n_periods)

s1_balance[0] = s1_opening

for i in range(1, n_periods):
    opening_bal = s1_balance[i-1]
    
    # Compute interest rate for this period
    rate = compute_s1_interest_rate(period_starts[i], period_ends[i], s1_rate_schedule)
    s1_interest_rate[i] = rate
    
    # Interest payment
    interest = opening_bal * rate
    s1_interest_payment[i] = interest
    
    # Principal repayment (only during repayment period, i.e., periods 1-12)
    if i <= s1_n_repayments and opening_bal > 0:
        repay = min(s1_principal_repayment, opening_bal)
        s1_principal_payment[i] = repay
    
    s1_balance[i] = opening_bal - s1_principal_payment[i]

# Question 1: Interest rate for Senior 1 in period ending Dec 31, 2019
# Period ending Dec 31, 2019 is period index 4 (Jul 1 2019 to Dec 31 2019)
q1_rate = s1_interest_rate[4]
print(f'Q1: Senior 1 interest rate for period ending Dec 31, 2019 = {q1_rate*100:.2f}%')

# Question 2: Total interest paid for Senior 1
q2_total_interest = np.sum(s1_interest_payment)
print(f'Q2: Total Senior 1 interest = ${q2_total_interest:,.2f}')
print(f'    Rounded to nearest dollar = ${round(q2_total_interest)}')

In [ ]:
# =====================================================
# SENIOR 2 LOAN
# =====================================================
# Opening balance: $10,000,000
# Interest rate: 6% p.a., semi-annual = 3%
# Until Dec 31, 2023: interest is capitalized (added to balance)
# From Jan 1, 2024: interest is paid as incurred
# From Jun 2024: principal repayments of $1,000,000 or outstanding balance

s2_opening = 10_000_000.0
s2_annual_rate = 0.06
s2_semi_rate = s2_annual_rate / 2  # 3%
s2_repayment_amount = 1_000_000.0
# First repayment date: Jun 30, 2024 = period index 13
# Interest capitalized until Dec 31, 2023 = period index 12

s2_balance = np.zeros(n_periods)
s2_interest_payment = np.zeros(n_periods)  # interest actually paid (cash)
s2_interest_capitalized = np.zeros(n_periods)  # interest added to balance
s2_principal_payment = np.zeros(n_periods)

s2_balance[0] = s2_opening

for i in range(1, n_periods):
    opening_bal = s2_balance[i-1]
    interest = opening_bal * s2_semi_rate
    
    # Determine if interest is capitalized or paid
    # Capitalize until Dec 31, 2023 (period 12)
    if i <= 12:
        # Capitalize interest
        s2_interest_capitalized[i] = interest
        s2_interest_payment[i] = 0
        new_bal = opening_bal + interest
    else:
        # Pay interest
        s2_interest_payment[i] = interest
        s2_interest_capitalized[i] = 0
        new_bal = opening_bal
    
    # Principal repayment from Jun 2024 (period 13) onwards
    if i >= 13:
        repay = min(s2_repayment_amount, new_bal)
        s2_principal_payment[i] = repay
        new_bal -= repay
    
    s2_balance[i] = new_bal

# Find the balance after capitalization (Dec 31, 2023)
print(f'Senior 2 balance at Dec 31, 2023 (after capitalization): ${s2_balance[12]:,.2f}')

# Question 3: Final repayment amount for Senior 2
# Find the last period where a principal repayment is made
last_repayment_idx = 0
for i in range(n_periods-1, 0, -1):
    if s2_principal_payment[i] > 0:
        last_repayment_idx = i
        break

q3_final_repayment = s2_principal_payment[last_repayment_idx]
print(f'Q3: Final Senior 2 repayment = ${q3_final_repayment:,.2f} at period {last_repayment_idx} ({period_ends[last_repayment_idx]})')

# Print Senior 2 balance schedule around the final repayment
for i in range(12, min(last_repayment_idx+3, n_periods)):
    print(f'  Period {i} ({period_ends[i]}): bal={s2_balance[i]:,.2f}, interest_paid={s2_interest_payment[i]:,.2f}, principal={s2_principal_payment[i]:,.2f}')

In [ ]:
# =====================================================
# MAJOR MAINTENANCE COSTS
# =====================================================
# Extract from CSV row 47 (0-indexed: row 46), columns 12 onward
# These are costs per semi-annual period

maintenance_costs_raw = [
    73092.09792639977, 26820.5671321037, 8703.41491647848, 95829.48798072721,
    55271.29321111428, 33716.89319239687, 23495.810908215943, 49706.545477399246,
    4413.153084847599, 58479.120545451435, 54721.72507647711, 73752.41742634523,
    7935.6963672138845, 49279.50498107708, 87670.46892570533, 9586.616463892828,
    80961.83099680748, 91064.35367227667, 65837.33363289462, 11925.239216300331,
    74666.54056604218, 92820.3648314648, 50698.87512938874, 67231.20507477636,
    30120.496947803975, 24843.132594042072, 88616.89616336337, 69638.4313800627,
    12577.035504995481, 52377.857345159064, 0, 0, 0, 0, 0, 0
]

# maintenance_costs[i] = costs in period i (i=1 is Jun 2018, i=36 is Dec 2035)
maintenance_costs = np.zeros(n_periods)
for i, cost in enumerate(maintenance_costs_raw):
    maintenance_costs[i+1] = cost  # period 1 = Jun 2018

print('Maintenance costs loaded for', len(maintenance_costs_raw), 'periods')
print('First few:', maintenance_costs[1:6])

In [ ]:
# =====================================================
# TARGET DSRA BALANCE
# =====================================================
# DSRA target = total senior debt interest and repayments due in the NEXT semi-annual period
# "next" means looking forward one period

# Total senior debt service (interest + principal) for each period
senior_debt_service = np.zeros(n_periods)
for i in range(n_periods):
    senior_debt_service[i] = (s1_interest_payment[i] + s1_principal_payment[i] + 
                              s2_interest_payment[i] + s2_principal_payment[i])

# Target DSRA at end of period i = debt service due in period i+1
dsra_target = np.zeros(n_periods)
for i in range(n_periods):
    if i + 1 < n_periods:
        dsra_target[i] = senior_debt_service[i+1]
    else:
        dsra_target[i] = 0  # No next period

# Question 4: Target DSRA balance on Jun 30, 2023 = period index 11
q4_dsra_target = dsra_target[11]
print(f'Q4: Target DSRA balance on Jun 30, 2023 = ${q4_dsra_target:,.2f}')

# Print surrounding periods for verification
for i in range(10, 15):
    print(f'  Period {i} ({period_ends[i]}): S1_int={s1_interest_payment[i]:,.2f}, S1_prin={s1_principal_payment[i]:,.2f}, S2_int={s2_interest_payment[i]:,.2f}, S2_prin={s2_principal_payment[i]:,.2f}, debt_svc={senior_debt_service[i]:,.2f}, DSRA_target={dsra_target[i]:,.2f}')

In [ ]:
# =====================================================
# TARGET MMRA BALANCE
# =====================================================
# Target MMRA balance at end of period i:
# 100% of next 12 months' maintenance costs
# + 66% of costs in 12 months after that
# + 33% of costs in 12 months after that
#
# Since periods are semi-annual (6 months each):
# Next 12 months = next 2 periods (i+1, i+2)
# Following 12 months = periods i+3, i+4
# Following 12 months = periods i+5, i+6

mmra_target = np.zeros(n_periods)
for i in range(n_periods):
    # Next 12 months: periods i+1, i+2
    next_12 = 0
    for j in [i+1, i+2]:
        if j < n_periods:
            next_12 += maintenance_costs[j]
    
    # 12-24 months: periods i+3, i+4
    next_12_24 = 0
    for j in [i+3, i+4]:
        if j < n_periods:
            next_12_24 += maintenance_costs[j]
    
    # 24-36 months: periods i+5, i+6
    next_24_36 = 0
    for j in [i+5, i+6]:
        if j < n_periods:
            next_24_36 += maintenance_costs[j]
    
    mmra_target[i] = 1.0 * next_12 + 0.66 * next_12_24 + 0.33 * next_24_36

# Question 5: Target MMRA balance on Dec 31, 2020 = period index 6
q5_mmra_target = mmra_target[6]
print(f'Q5: Target MMRA balance on Dec 31, 2020 = ${q5_mmra_target:,.2f}')

# Print surrounding periods for verification
for i in range(5, 9):
    print(f'  Period {i} ({period_ends[i]}): maintenance_cost={maintenance_costs[i]:,.2f}, MMRA_target={mmra_target[i]:,.2f}')

In [ ]:
# =====================================================
# SHAREHOLDER LOAN
# =====================================================
# Opening balance: $3,000,000
# Interest: 9% p.a., semi-annual rate = 4.5%
# Target balance: annuity that is fully repaid by Dec 31, 2030
# Payments from Jun 2018 to Dec 2030 = 26 semi-annual periods

sh_opening = 3_000_000.0
sh_annual_rate = 0.09
sh_semi_rate = sh_annual_rate / 2  # 4.5%

# Number of annuity payments: Jun 2018 (period 1) through Dec 2030 (period 26) = 26 periods
sh_n_payments = 26

# Annuity payment amount (PMT formula)
# PMT = PV * r / (1 - (1+r)^(-n))
r = sh_semi_rate
n = sh_n_payments
sh_annuity_payment = sh_opening * r / (1 - (1 + r)**(-n))
print(f'Shareholder annuity payment per period: ${sh_annuity_payment:,.2f}')

# Compute target shareholder loan balance at end of each period
# The target balance is what the balance would be under the annuity schedule
sh_target_balance = np.zeros(n_periods)
sh_target_balance[0] = sh_opening  # Initial balance

sh_target_interest = np.zeros(n_periods)
sh_target_principal = np.zeros(n_periods)

for i in range(1, n_periods):
    prev_bal = sh_target_balance[i-1]
    if prev_bal > 0 and i <= 26:  # Payments from period 1 to 26
        interest = prev_bal * sh_semi_rate
        sh_target_interest[i] = interest
        principal = sh_annuity_payment - interest
        sh_target_principal[i] = principal
        sh_target_balance[i] = prev_bal - principal
    else:
        sh_target_balance[i] = 0

# Question 6: Target shareholder loan balance on Jun 30, 2025 = period 15
q6_sh_target = sh_target_balance[15]
print(f'Q6: Target shareholder loan balance on Jun 30, 2025 = ${q6_sh_target:,.2f}')

# Print surrounding periods
for i in range(13, 18):
    print(f'  Period {i} ({period_ends[i]}): target_bal={sh_target_balance[i]:,.2f}, interest={sh_target_interest[i]:,.2f}, principal={sh_target_principal[i]:,.2f}')

In [ ]:
# =====================================================
# CASH PROFILES FROM OPERATIONS
# =====================================================

profiles_raw = {
    1: [1193084.5839555846,1280899.3484749948,1237461.5561064575,1181674.7928202343,1187108.2130608882,1207601.4330362824,1170681.5042149932,1182884.7987555645,1153305.256473944,1100499.9034176273,1105157.8017220697,1616770.0407034103,1713948.721569852,1573108.9147265651,1559993.9216692278,1591703.0918475748,1467319.8935123894,1451971.811436218,1435034.5963928928,1476460.12628166,1379371.05111941,1304179.6397521265,1323636.1917217178,1265138.089827525,1248369.694648682,505444.79650658526,-57091.62641549735,-31203.811956714235,-8819.314869349373,-46128.30191589506,8245.31603029411,25782.66551455516,34752.99111466158,32998.080163971,42297.494895356744,40214.70352590203],
    2: [1180270.5100223545,1275727.9518326232,1221797.6980047529,1167161.3141350483,1180871.4278817028,1184313.8668762972,1154687.2609332239,1135398.2544205307,1135044.6594328408,1095546.5670304229,1103562.6892127974,1609862.4970491393,1710808.4985140455,1572929.3797641397,1527039.8074057612,1554812.6776954797,1466280.4289328577,1440321.0663587325,1385475.196329675,1428040.7712023212,1375207.8706473806,1281561.8972717102,1311073.5811941375,1256117.9308699076,1238019.6035925255,495866.46626321157,-29993.434084245728,-17262.397536086624,-5248.968283369812,-29309.831868909387,5364.199311935145,23623.771762550583,19769.98545361767,13100.695255903855,37471.120351427366,24798.898156708856],
    3: [1009907.7767296777,1086044.7424740118,1063390.8068090295,1002073.2850308537,1034436.4353032059,1056688.690667834,986173.9847669421,986153.9086161447,990685.8230284784,918095.0884897148,893797.0034268994,840035.113144301,1544398.1844579834,1396383.5891481957,1395650.5320606618,1405830.044246416,1340335.6482896123,1287750.1281211637,1278460.4382966105,1277028.569937704,1170168.7465947806,1126329.9104620805,1157864.7610814986,1119267.8543583266,1087927.5954720874,1073242.9433401334,195914.12794937385,28799.55482856933,32950.44907603219,12413.768329773156,28755.32681122821,7332.232820395535,22907.039652855277,36703.67065178551,44667.206307779976,21744.17382170701],
    4: [1135914.5848578413,1079551.501252565,1108888.507993919,1060065.3995099235,1027593.7394839926,1065199.3937953522,1012805.6512353873,1052042.8016104808,948495.3112838411,973654.2678233148,901381.7192541864,894081.6556886411,1501235.5490796845,1461383.4562511456,1456460.0173745945,1379689.4306617412,1377222.963654068,1368694.3402378894,1249473.3883326333,1224151.6278346197,1259019.5222129412,1208819.8621638087,1147389.9984873058,1196583.9062468598,1158072.2619812062,1130380.5158643872,335192.69359270105,77694.23619259014,269.9055401925943,9544.464130807073,78501.80343073803,55367.969795954894,41052.94836009159,50586.88842391499,20800.062561315648,57805.464488113656],
    5: [1174051.3997789517,1265415.18272047,1229785.8493751897,168009.70530445,179623.06660843,204700.09538344,1161914.4151174228,1180132.864521018,1145362.5218711228,1090878.1832208675,1101025.3946864046,1604454.2182697374,1701948.095096875,1564498.88512978,1559754.7159481517,1584101.2969582938,1459009.3090321948,1449926.2675272017,1434990.5470021337,1457678.0936877418,1376200.725276786,1293915.889738651,1321035.9229311743,1256085.8744126938,1248037.5256264748,495408.9518856897,-39272.16798996067,-13751.723354345017,-3614.114906584882,-25825.55925580496,2795.128402670729,12721.911674128265,19670.732147701172,17621.17830227495,27952.233739057305,25560.384182257338]
}

# Build cash arrays (period 0 = 0, periods 1-36 = cash from operations)
cash_profiles = {}
for p_num, vals in profiles_raw.items():
    cash = np.zeros(n_periods)
    for i, v in enumerate(vals):
        cash[i+1] = v
    cash_profiles[p_num] = cash

print('Cash profiles loaded for profiles 1-5')

In [ ]:
# =====================================================
# PRIORITY OF PAYMENTS (WATERFALL) MODEL
# =====================================================

def run_waterfall(profile_num):
    """
    Run the full priority of payments model for a given cash profile.
    Returns a dict with all account balances and payment arrays.
    """
    cash_from_ops = cash_profiles[profile_num].copy()
    
    # Initialize arrays
    dsra_balance = np.zeros(n_periods)  # DSRA account balance
    mmra_balance = np.zeros(n_periods)  # MMRA account balance
    overdraft = np.zeros(n_periods)     # Overdraft balance (positive = owed)
    
    # Shareholder loan actual balance
    sh_actual_balance = np.zeros(n_periods)
    sh_interest_paid = np.zeros(n_periods)
    sh_principal_paid = np.zeros(n_periods)
    sh_interest_capitalized = np.zeros(n_periods)
    
    dividends = np.zeros(n_periods)
    
    # Initial balances at period 0 (Dec 31, 2017)
    dsra_balance[0] = dsra_target[0]  # Fully funded
    mmra_balance[0] = mmra_target[0]  # Fully funded
    sh_actual_balance[0] = sh_opening
    overdraft[0] = 0
    
    for i in range(1, n_periods):
        # Step 0: Determine available cash
        # Start with cash from operations
        available = cash_from_ops[i]
        
        # Release excess from DSRA and MMRA
        # DSRA: if opening balance > current period's target, release excess
        # Wait - the instruction says: "If the balances of the DSRA and MMRA at the 
        # start of the semi-annual period exceed the target balance for the semi-annual period"
        # The target balance "for the semi-annual period" = dsra_target[i] (target at end of this period)
        dsra_opening = dsra_balance[i-1]
        mmra_opening = mmra_balance[i-1]
        
        dsra_excess = max(0, dsra_opening - dsra_target[i])
        mmra_excess = max(0, mmra_opening - mmra_target[i])
        
        available += dsra_excess + mmra_excess
        dsra_current = dsra_opening - dsra_excess
        mmra_current = mmra_opening - mmra_excess
        
        # Step 1: Pay senior debt (interest + principal)
        total_senior_service = senior_debt_service[i]
        
        if available >= total_senior_service:
            available -= total_senior_service
        else:
            shortfall = total_senior_service - available
            available = 0
            
            # Draw from DSRA
            dsra_draw = min(shortfall, dsra_current)
            dsra_current -= dsra_draw
            shortfall -= dsra_draw
            
            # Draw from MMRA
            if shortfall > 0:
                mmra_draw = min(shortfall, mmra_current)
                mmra_current -= mmra_draw
                shortfall -= mmra_draw
            
            # Draw overdraft
            if shortfall > 0:
                overdraft[i] = overdraft[i-1] + shortfall
                shortfall = 0
            else:
                overdraft[i] = overdraft[i-1]
        
        if available > 0:  # Only set overdraft from previous if we didn't draw new
            if overdraft[i] == 0:  # Wasn't set above
                overdraft[i] = overdraft[i-1]
        elif overdraft[i] == 0:
            overdraft[i] = overdraft[i-1]
        
        # Step 2: Repay overdraft
        if available > 0 and overdraft[i] > 0:
            od_repay = min(available, overdraft[i])
            available -= od_repay
            overdraft[i] -= od_repay
        
        # Step 3: Fund MMRA up to target
        mmra_shortfall = max(0, mmra_target[i] - mmra_current)
        if available > 0 and mmra_shortfall > 0:
            mmra_fund = min(available, mmra_shortfall)
            mmra_current += mmra_fund
            available -= mmra_fund
        
        # Step 4: Fund DSRA up to target
        dsra_shortfall = max(0, dsra_target[i] - dsra_current)
        if available > 0 and dsra_shortfall > 0:
            dsra_fund = min(available, dsra_shortfall)
            dsra_current += dsra_fund
            available -= dsra_fund
        
        # Step 5: Pay shareholder interest
        sh_prev_bal = sh_actual_balance[i-1]
        sh_interest_due = sh_prev_bal * sh_semi_rate
        
        if available >= sh_interest_due:
            sh_interest_paid[i] = sh_interest_due
            available -= sh_interest_due
            sh_interest_capitalized[i] = 0
        else:
            sh_interest_paid[i] = available
            sh_interest_capitalized[i] = sh_interest_due - available
            available = 0
        
        # Update shareholder balance with any capitalized interest
        sh_bal_after_interest = sh_prev_bal + sh_interest_capitalized[i]
        
        # Step 6: Pay off shareholder loan exceeding target balance
        sh_target_this_period = sh_target_balance[i]
        sh_excess = max(0, sh_bal_after_interest - sh_target_this_period)
        
        if available > 0 and sh_excess > 0:
            sh_repay = min(available, sh_excess)
            sh_principal_paid[i] = sh_repay
            available -= sh_repay
            sh_actual_balance[i] = sh_bal_after_interest - sh_repay
        else:
            sh_actual_balance[i] = sh_bal_after_interest
        
        # Remaining cash = dividend
        if available > 0:
            dividends[i] = available
        
        # Set final account balances
        dsra_balance[i] = dsra_current
        mmra_balance[i] = mmra_current
    
    return {
        'dsra_balance': dsra_balance,
        'mmra_balance': mmra_balance,
        'overdraft': overdraft,
        'sh_actual_balance': sh_actual_balance,
        'sh_interest_paid': sh_interest_paid,
        'sh_principal_paid': sh_principal_paid,
        'sh_interest_capitalized': sh_interest_capitalized,
        'dividends': dividends,
    }

print('Waterfall model defined')

In [ ]:
# =====================================================
# RUN ALL PROFILES AND ANSWER QUESTIONS
# =====================================================

results = {}
for p in range(1, 6):
    results[p] = run_waterfall(p)
    print(f'Profile {p}: Total dividends = ${np.sum(results[p]["dividends"]):,.2f}')

print()

In [ ]:
# =====================================================
# ANSWER ALL QUESTIONS
# =====================================================

# Question 1: Interest rate for Senior 1 in period ending Dec 31, 2019
# Period: Jul 1, 2019 to Dec 31, 2019 (period index 4)
q1_rate_pct = s1_interest_rate[4] * 100
print(f'Q1: Senior 1 rate for period ending Dec 31, 2019 = {q1_rate_pct:.2f}%')

# Map to multiple choice
q1_options = {'A': 2.21, 'B': 2.22, 'C': 2.23, 'D': 2.24, 'E': 2.25, 'F': 2.26, 'G': 2.27, 'H': 2.28, 'I': 2.29}
q1_answer = min(q1_options, key=lambda k: abs(q1_options[k] - q1_rate_pct))
print(f'  -> Answer: {q1_answer}')

# Question 2: Total interest paid for Senior 1
q2_total = round(np.sum(s1_interest_payment))
print(f'Q2: Total Senior 1 interest = ${q2_total:,}')
q2_options = {'A': 1613196, 'B': 1613197, 'C': 1613198, 'D': 1613199, 'E': 1613200, 'F': 1613201, 'G': 1613202, 'H': 1613203, 'I': 1613204}
q2_answer = min(q2_options, key=lambda k: abs(q2_options[k] - q2_total))
print(f'  -> Answer: {q2_answer}')

# Question 3: Final repayment amount for Senior 2
q3_val = round(q3_final_repayment)
print(f'Q3: Final Senior 2 repayment = ${q3_val:,}')
q3_options = {'A': 257601, 'B': 257602, 'C': 257603, 'D': 257604, 'E': 257605, 'F': 257606, 'G': 257607, 'H': 257608, 'I': 257609}
q3_answer = min(q3_options, key=lambda k: abs(q3_options[k] - q3_val))
print(f'  -> Answer: {q3_answer}')

# Question 4: Target DSRA balance on Jun 30, 2023
q4_val = round(q4_dsra_target)
print(f'Q4: Target DSRA on Jun 30, 2023 = ${q4_val:,}')
q4_options = {'A': 858318, 'B': 858319, 'C': 858320, 'D': 858321, 'E': 858322, 'F': 858323, 'G': 858324, 'H': 858325, 'I': 858326}
q4_answer = min(q4_options, key=lambda k: abs(q4_options[k] - q4_val))
print(f'  -> Answer: {q4_answer}')

# Question 5: Target MMRA balance on Dec 31, 2020
q5_val = round(q5_mmra_target)
print(f'Q5: Target MMRA on Dec 31, 2020 = ${q5_val:,}')
q5_options = {'A': 157104, 'B': 157105, 'C': 157106, 'D': 157107, 'E': 157108, 'F': 157109, 'G': 157110, 'H': 157111, 'I': 157112}
q5_answer = min(q5_options, key=lambda k: abs(q5_options[k] - q5_val))
print(f'  -> Answer: {q5_answer}')

# Question 6: Target shareholder loan balance on Jun 30, 2025
q6_val = round(q6_sh_target)
print(f'Q6: Target SH loan on Jun 30, 2025 = ${q6_val:,}')
q6_options = {'A': 1689272, 'B': 1689273, 'C': 1689274, 'D': 1689276, 'E': 1689275, 'F': 1689277, 'G': 1689278, 'H': 1689279, 'I': 1689280}
q6_answer = min(q6_options, key=lambda k: abs(q6_options[k] - q6_val))
print(f'  -> Answer: {q6_answer}')

# Question 7: Profile 1 total dividends
q7_val = round(np.sum(results[1]['dividends']))
print(f'Q7: Profile 1 total dividends = ${q7_val:,}')
q7_options = {'A': 914772, 'B': 914773, 'C': 914774, 'D': 914775, 'E': 914776, 'F': 914777, 'G': 914778, 'H': 914779, 'I': 914780}
q7_answer = min(q7_options, key=lambda k: abs(q7_options[k] - q7_val))
print(f'  -> Answer: {q7_answer}')

# Question 8: Profile 2 final repayment for shareholder loan
# Find the last non-zero shareholder principal payment
sh_payments_p2 = results[2]['sh_principal_paid']
last_sh_repay_idx = 0
for i in range(n_periods-1, 0, -1):
    if sh_payments_p2[i] > 0.01:
        last_sh_repay_idx = i
        break
q8_val = round(sh_payments_p2[last_sh_repay_idx])
print(f'Q8: Profile 2 final SH repayment = ${q8_val:,} (period {last_sh_repay_idx}, {period_ends[last_sh_repay_idx]})')
q8_options = {'A': 191462, 'B': 191463, 'C': 191464, 'D': 191465, 'E': 191466, 'F': 191467, 'G': 191468, 'H': 191469, 'I': 191470}
q8_answer = min(q8_options, key=lambda k: abs(q8_options[k] - q8_val))
print(f'  -> Answer: {q8_answer}')

# Question 9: Profile 3, how many periods is DSRA underfunded?
dsra_underfunded_count = 0
for i in range(1, n_periods):
    if results[3]['dsra_balance'][i] < dsra_target[i] - 0.005:
        dsra_underfunded_count += 1
q9_val = dsra_underfunded_count
print(f'Q9: Profile 3 DSRA underfunded periods = {q9_val}')
q9_options = {'A': 4, 'B': 5, 'C': 6, 'D': 7, 'E': 8, 'F': 9, 'G': 10, 'H': 11, 'I': 12}
q9_answer = min(q9_options, key=lambda k: abs(q9_options[k] - q9_val))
print(f'  -> Answer: {q9_answer}')

# Question 10: Profile 4 max MMRA underfunding
mmra_underfunding = np.zeros(n_periods)
for i in range(n_periods):
    mmra_underfunding[i] = max(0, mmra_target[i] - results[4]['mmra_balance'][i])
q10_val = round(np.max(mmra_underfunding))
print(f'Q10: Profile 4 max MMRA underfunding = ${q10_val:,}')
q10_options = {'A': 13980, 'B': 13981, 'C': 13982, 'D': 13983, 'E': 13984, 'F': 13985, 'G': 13986, 'H': 13987, 'I': 13988}
q10_answer = min(q10_options, key=lambda k: abs(q10_options[k] - q10_val))
print(f'  -> Answer: {q10_answer}')

# Question 11: Profile 5 max overdraft balance
q11_val = round(np.max(results[5]['overdraft']))
print(f'Q11: Profile 5 max overdraft = ${q11_val:,}')
q11_options = {'A': 1252176, 'B': 1252177, 'C': 1252178, 'D': 1252179, 'E': 1252180, 'F': 1252181, 'G': 1252182, 'H': 1252183, 'I': 1252184}
q11_answer = min(q11_options, key=lambda k: abs(q11_options[k] - q11_val))
print(f'  -> Answer: {q11_answer}')

In [ ]:
# =====================================================
# FINAL ANSWERS
# =====================================================

answers = {
    "question1": q1_answer,
    "question2": q2_answer,
    "question3": q3_answer,
    "question4": q4_answer,
    "question5": q5_answer,
    "question6": q6_answer,
    "question7": q7_answer,
    "question8": q8_answer,
    "question9": q9_answer,
    "question10": q10_answer,
    "question11": q11_answer,
}

print('Final answers:')
for k, v in answers.items():
    print(f'  {k}: {v}')